# Model 3 - Local Execution

This notebook implements the training and inference pipeline for the mailing time prediction model. It includes fixes for XGBoost compatibility and enhanced analysis tools.

In [ ]:
import numpy as np
import pandas as pd
import xgboost as xgb
import pickle
import onnxmltools
import hashlib
import os
import time
import gc
import onnxruntime as ort
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score
from onnxmltools.convert.common.data_types import FloatTensorType
from tqdm import tqdm

## Configuration

In [ ]:
CSV_FILENAME = "nwl3_mailer_processed_view.csv"
MODEL_PKL_FILENAME = "xgboost_model.pkl"
MODEL_ONNX_FILENAME = "xgboost_model.onnx"
INFERENCE_OUTPUT_FILENAME = "inference_FULL_PROD_model.csv"

NUM_CLASSES = 10
SAMPLE_THRESHOLD = 15_000_000
IDX_TO_HOUR = {
    0: 7, 1: 8, 2: 9, 3: 10, 4: 11,
    5: 12, 6: 15, 7: 16, 8: 19, 9: 20
}
CLASS_MAP = IDX_TO_HOUR

FEATURE_COLS = [
    "sent_hour", "sent_dow", "sent_day", "sent_month",
    "is_weekend", "hour_squared", "time_to_open",
    "web_id", "client_id"
]

## Data Loading

In [ ]:
def load_and_prepare_data(csv_path):
    print(f"\n[1/8] Loading Data from {csv_path}...")
    
    use_cols = [
        'sent', 'opened',
        'web_id', 'client_id'
    ]
    
    if not os.path.exists(csv_path) and os.path.exists(csv_path + ".zip"):
        csv_path += ".zip"
        print(f"  -> Found zip: {csv_path}")

    # Configure for Full Dataset
    N_ROWS = None 
    CHUNK_SIZE = 500_000
    
    chunks = []
    
    print(f"  -> Reading CSV in chunks of {CHUNK_SIZE:,} to optimize memory...")
    
    # Iterate through chunks to filter early
    with pd.read_csv(
        csv_path, 
        usecols=use_cols,
        nrows=N_ROWS,
        chunksize=CHUNK_SIZE,
        low_memory=True
    ) as reader:
        for chunk in tqdm(reader, desc="Loading & Filtering Chunks"):
            # Filter: we only care about opened emails for training
            chunk_filtered = chunk.dropna(subset=['opened']).copy()
            if not chunk_filtered.empty:
                chunks.append(chunk_filtered)
    
    if not chunks:
        raise ValueError("No valid data found (no opened emails).")

    # Concatenate all valid parts
    df = pd.concat(chunks, ignore_index=True)
    print(f"  -> Loaded {len(df):,} rows with 'opened' populated.")
    
    # Free up memory
    del chunks
    gc.collect()
    
    print("  -> Parsing datetimes...")
    df['sent_dt'] = pd.to_datetime(df['sent'], errors='coerce')
    df['opened_dt'] = pd.to_datetime(df['opened'], errors='coerce')
    
    df = df.dropna(subset=['sent_dt', 'opened_dt'])
    
    print("  -> Generating targets...")
    df['open_hour'] = df['opened_dt'].dt.hour
    
    target_hours = list(IDX_TO_HOUR.values())
    df = df[df['open_hour'].isin(target_hours)].copy()
    
    HOUR_TO_IDX = {v: k for k, v in IDX_TO_HOUR.items()}
    df['target'] = df['open_hour'].map(HOUR_TO_IDX).astype(int)
    
    print(f"  -> Rows after filtering target hours: {len(df):,}")
    
    print("  -> Generating features...")
    df["sent_hour"] = df["sent_dt"].dt.hour.astype("int8")
    df["sent_dow"] = df["sent_dt"].dt.dayofweek.astype("int8")
    df["sent_day"] = df["sent_dt"].dt.day.astype("int8")
    df["sent_month"] = df["sent_dt"].dt.month.astype("int8")
    df["is_weekend"] = (df["sent_dow"] >= 5).astype("int8")
    df["hour_squared"] = (df["sent_hour"] ** 2).astype("int16")
    df["time_to_open"] = 2.0
    
    if "web_id" not in df.columns: df["web_id"] = 0
    if "client_id" not in df.columns: df["client_id"] = 0
    
    df["web_id"] = df["web_id"].fillna(0).astype(int)
    df["client_id"] = df["client_id"].fillna(0).astype(int)

    X = df[FEATURE_COLS]
    y = df['target']
    
    print(f"  -> Final Dataset: {X.shape}")
    return X, y

## Training Logic

In [ ]:
def train_model(X, y):
    print("\n[3/8] Train/Test Split")
    print("-" * 80)
    
    # Standard 80/20 Split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.20,
        random_state=42,
        stratify=y
    )
    
    print(f"Train: {X_train.shape[0]:,} | Test: {X_test.shape[0]:,}")
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train).astype(np.float32)
    X_test_scaled = scaler.transform(X_test).astype(np.float32)
    
    gc.collect()
    
    print("Calculating class weights...")
    ytrain_series = pd.Series(y_train)
    class_counts = ytrain_series.value_counts().sort_index()
    total = len(ytrain_series)
    class_weights = (total / (len(class_counts) * class_counts)).to_dict()
    sample_weights = ytrain_series.map(class_weights).values

    print("\n[4/8] XGBoost Training (IMPROVED HYPERPARAMETERS)")
    print("-" * 80)
    
    dataset_size = len(X_train)
    xgb_success = False
    start_time = time.time()
    xgb_model = None

    # Force Level 1 logic for full dataset training
    print(f"Level 1: Full dataset ({dataset_size:,} samples)")
    try:
        xgb_model = xgb.XGBClassifier(
            objective='multi:softmax',
            num_class=NUM_CLASSES,
            max_depth=4,
            n_estimators=60,
            learning_rate=0.15,
            subsample=0.8,
            colsample_bytree=0.8,
            colsample_bylevel=0.8,
            tree_method='hist',
            grow_policy='lossguide',
            max_leaves=48,
            max_bin=256,
            min_child_weight=1,
            gamma=0,
            reg_alpha=0.1,
            reg_lambda=1.0,
            random_state=42,
            n_jobs=-1, # Use all available cores
            early_stopping_rounds=15, 
            eval_metric="mlogloss",
            verbosity=1 # Show progress
        )
        
        eval_size = min(100000, len(X_test))
        xgb_model.fit(
            X_train, y_train,
            sample_weight=sample_weights,
            eval_set=[(X_test[:eval_size], y_test[:eval_size])],
            verbose=True
        )
        
        y_pred = xgb_model.predict(X_test)
        acc = accuracy_score(y_test, y_pred)
        print(f"Level 1 successful. Time: {time.time()-start_time:.1f}s")
        print(f"Accuracy: {acc*100:.2f}%")
        xgb_success = True
        
    except Exception as e:
        print(f"Level 1 failed: {str(e)}")

    if xgb_success:
        f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
        print(f"F1-score: {f1:.4f}")
        
        print("\nTop 5 features:")
        importance = xgb_model.feature_importances_
        sorted_idx = np.argsort(importance)[-5:][::-1]
        for i, idx in enumerate(sorted_idx, 1):
            if idx < len(FEATURE_COLS):
                 print(f"  {i}. {FEATURE_COLS[idx]:<20} {importance[idx]:>6.4f}")
        
        return xgb_model, X_test, y_test
    return None, None, None

## Evaluation & Export

In [ ]:
def evaluate_blocks(model, X_test, y_test):
    print("\n[7/8] Block-wise Performance")
    print("-" * 80)
    
    y_pred = model.predict(X_test)
    
    y_test_hours = np.array([IDX_TO_HOUR[i] for i in y_test])
    y_pred_hours = np.array([IDX_TO_HOUR[i] for i in y_pred])
    
    blocks = [
        ("7-8h", [7, 8]),
        ("9-10h", [9, 10]),
        ("11-12h", [11, 12]),
        ("15-16h", [15, 16]),
        ("19-20h", [19, 20])
    ]
    
    print(f"{'Block':<12} {'Samples':<10} {'Accuracy':<10}")
    print("-" * 35)
    
    for block_name, hours in blocks:
        mask = np.isin(y_test_hours, hours)
        if np.sum(mask) > 0:
            block_acc = accuracy_score(y_test_hours[mask], y_pred_hours[mask])
            print(f"{block_name:<12} {np.sum(mask):<10,} {block_acc*100:>7.2f}%")

def analyze_predictions(model, X_test):
    """
    Analyzes predictions on the test set (unseen data).
    Plots appropriate histograms and prints a distribution table.
    """
    print("\n[8/8] Prediction Distribution Analysis")
    print("-" * 80)
    
    # Predict
    y_pred_idx = model.predict(X_test)
    y_pred_hours = [IDX_TO_HOUR[i] for i in y_pred_idx]
    
    # Create DataFrame for analysis
    pred_df = pd.DataFrame({'pred_hour': y_pred_hours})
    
    # Calculate counts and percentages
    counts = pred_df['pred_hour'].value_counts().sort_index()
    percentages = (counts / len(pred_df)) * 100
    
    summary_df = pd.DataFrame({
        'count': counts,
        'percent': percentages.round(2)
    })
    summary_df.index.name = 'pred_hour'
    summary_df = summary_df.reset_index()
    
    print("Tabulka rozdělení predikcí:")
    print(summary_df.to_string(index=False))
    
    # Plot
    plt.figure(figsize=(10, 6))
    plt.bar(summary_df['pred_hour'].astype(str), summary_df['count'], color='skyblue', edgecolor='black')
    plt.xlabel('Predicted Hour')
    plt.ylabel('Count')
    plt.title('Distribution of Predicted Best Send Times (Test Set)')
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.show()

def export_to_onnx(model, n_features=9):
    print(f"\nExporting model to {MODEL_ONNX_FILENAME}...")
    
    # FIX: onnxmltools can crash with named features (ValueError: could not convert string to float: 'hour_squared')
    # We strip feature names from the booster before export to ensure 'f0', 'f1' format
    try:
        model.get_booster().feature_names = None
    except Exception as e:
        print(f"Warning: Could not clear feature names: {e}")
    
    initial_type = [('float_input', FloatTensorType([None, n_features]))]
    onnx_model = onnxmltools.convert_xgboost(model, initial_types=initial_type)
    
    with open(MODEL_ONNX_FILENAME, "wb") as f:
        f.write(onnx_model.SerializeToString())
    print(f"✓ Export finished: {MODEL_ONNX_FILENAME}")

## Main Execution

In [ ]:
# 1. Load Data
X, y = load_and_prepare_data(CSV_FILENAME)

# 2. Train
xgb_model, X_test, y_test = train_model(X, y)

if xgb_model:
    # 3. Evaluate
    evaluate_blocks(xgb_model, X_test, y_test)
    
    # 4. Analyze Predictions (User Request)
    analyze_predictions(xgb_model, X_test)
    
    # 5. Save Pickle
    with open(MODEL_PKL_FILENAME, 'wb') as f:
        pickle.dump(xgb_model, f)
    print(f"Saved model to {MODEL_PKL_FILENAME}")
    
    # 6. Export ONNX
    export_to_onnx(xgb_model, n_features=len(FEATURE_COLS))